<a href="https://colab.research.google.com/github/safaabuzaid/mri-generalization/blob/main/04_MC_dropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/MRI_Generalization"

MODELS_PATH = os.path.join(PROJECT_PATH, "models")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project:", PROJECT_PATH)
print("Models:", MODELS_PATH)
print("Device:", DEVICE)

Project: /content/drive/MyDrive/MRI_Generalization
Models: /content/drive/MyDrive/MRI_Generalization/models
Device: cpu


In [ ]:
print("Checking project files...")

print(os.listdir(PROJECT_PATH))

Checking project files...
['project_data', '01_DataExploration.ipynb', 'src', 'models', 'Report.gdoc', 'results ', 'MRI_Generalization_project_log.docx', 'Paper.gdoc', 'MRI_Notebooks']


In [ ]:
SRC_PATH = os.path.join(PROJECT_PATH, "src")

print(os.listdir(SRC_PATH))

['__pycache__', 'models.py', 'evaluate.py', 'data_preparation.py', 'transform.py', 'data.py', 'train.py', 'mc_dropout.py']


In [ ]:
import sys

sys.path.append(SRC_PATH)

from models import get_effecientnet_b3
from mc_dropout import enable_mc_dropout, mc_dropout_predict, run_mc_dropout

print("EfficientNet-B3 function imported successfully!")

EfficientNet-B3 function imported successfully!


In [ ]:
model, criterion, optimizer, device = get_effecientnet_b3()

model = model.to(DEVICE)

print("EfficientNet-B3 model created.")
print("Device:", DEVICE)


EfficientNet-B3 model created.
Device: cpu


In [ ]:
checkpoint_path = os.path.join(
    MODELS_PATH,
    "efficientnet_b3_seed42_best.pth"
)

model.load_state_dict(
    torch.load(checkpoint_path, map_location=DEVICE)
)

model = model.to(DEVICE)

print("Checkpoint loaded successfully!")
print("Checkpoint:", checkpoint_path)

Checkpoint loaded successfully!
Checkpoint: /content/drive/MyDrive/MRI_Generalization/models/efficientnet_b3_seed42_best.pth


In [ ]:
from data import *
from data_preparation import *
from transform import *

print("Dataset functions imported.")

Dataset functions imported.


In [ ]:
print("data.py functions/objects:")
print([name for name in dir() if not name.startswith("_")])

data.py functions/objects:
['BrainTumorDataset', 'ConcatDataset', 'DEVICE', 'DataLoader', 'Dataset', 'Image', 'In', 'MODELS_PATH', 'Out', 'PROJECT_PATH', 'SRC_PATH', 'augmentation_transform', 'baseline_transform', 'checkpoint_path', 'create_dataloaders', 'create_dataset', 'criterion', 'device', 'drive', 'enable_mc_dropout', 'exit', 'get_effecientnet_b3', 'get_ipython', 'mc_dropout_predict', 'model', 'nn', 'np', 'optimizer', 'os', 'pd', 'quit', 'random_split', 'run_mc_dropout', 'split_dataset', 'sys', 'torch', 'transforms']


In [ ]:
import inspect

print(inspect.signature(create_dataset))


(train_path, test_path, transform)


In [ ]:
print(inspect.getsource(create_dataset))


def create_dataset (train_path, test_path, transform):
  dataB_train = BrainTumorDataset(train_path,transform=transform)
  dataB_test = BrainTumorDataset(test_path,transform=transform)
  dataB = ConcatDataset([dataB_train,dataB_test])

  return dataB



In [ ]:
print(inspect.signature(BrainTumorDataset))


(dataframe, transform=None)


In [ ]:
print(inspect.getsource(BrainTumorDataset))


class BrainTumorDataset (Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        image_path = self.dataframe.loc[idx, "path"]
        label = int(self.dataframe.loc[idx, "label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label



In [ ]:
print([name for name in dir() if "data" in name.lower() or "df" in name.lower()])

['BrainTumorDataset', 'ConcatDataset', 'DataLoader', 'Dataset', 'create_dataloaders', 'create_dataset', 'split_dataset']


In [ ]:
print(inspect.getsource(create_dataloaders))


def create_dataloaders (train, val, test, batch_size = 32):
  train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val, batch_size=batch_size, shuffle=False)
  test_loader = DataLoader(test, batch_size=batch_size, shuffle=False)

  return train_loader, val_loader, test_loader



In [ ]:
TEST_SPLIT_PATH = os.path.join(
    PROJECT_PATH,
    "project_data",
    "splits",
    "test_split.csv"
)

test_df = pd.read_csv(TEST_SPLIT_PATH)

test_dataset = BrainTumorDataset(
    dataframe=test_df,
    transform=baseline_transform
)

print("External Test:", len(test_dataset))


External Test: 2543


In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Test loader created.")
print("Number of batches:", len(test_loader))


Test loader created.
Number of batches: 80


### Check where Dropout exists

In [ ]:
dropout_layers = []

for name, module in model.named_modules():
    if isinstance(module, nn.Dropout):
        dropout_layers.append(name)

print("Number of dropout layers:", len(dropout_layers))
print("Dropout layers:")
print(dropout_layers)

Number of dropout layers: 1
Dropout layers:
['classifier.0']


### Enable the dorpout

In [ ]:
model = enable_mc_dropout(model)

print("MC Dropout enabled.")

MC Dropout enabled.


In [ ]:
# Number of MC passes

MC_PASSES = 10

images, labels = next(iter(test_loader))

images = images.to(DEVICE)

with torch.no_grad():
    mean_probabilities, predicted_classes, uncertainty = mc_dropout_predict(
        model,
        images,
        mc_samples=MC_PASSES
    )

print("Batch size:", images.size(0))
print("Mean probabilities shape:", mean_probabilities.shape)
print("Predicted classes shape:", predicted_classes.shape)
print("Uncertainty shape:", uncertainty.shape)

Batch size: 32
Mean probabilities shape: torch.Size([32, 3])
Predicted classes shape: torch.Size([32])
Uncertainty shape: torch.Size([32])


In [22]:
mc_results = run_mc_dropout(
    model,
    test_loader,
    DEVICE,
    mc_passes=MC_PASSES
)

print("MC Dropout completed!")
print("Number of images:", len(mc_results))
print(mc_results.head())

Processed 10/80 batches
Processed 20/80 batches
Processed 30/80 batches
Processed 40/80 batches
Processed 50/80 batches
Processed 60/80 batches
Processed 70/80 batches
Processed 80/80 batches
MC Dropout completed!
Number of images: 2543
   image_id  true_class  predicted_class  prob_class_0  prob_class_1  \
0         0           0                0      0.999853      0.000128   
1         1           0                0      0.999956      0.000017   
2         2           0                0      0.999929      0.000034   
3         3           0                0      0.999974      0.000007   
4         4           0                0      0.999983      0.000008   

   prob_class_2  uncertainty  correct  
0      0.000019     0.001501        1  
1      0.000028     0.000518        1  
2      0.000037     0.000796        1  
3      0.000018     0.000312        1  
4      0.000009     0.000218        1  


In [23]:
print(test_df["label"].value_counts().sort_index())
print("\nUnique labels:")
print(sorted(test_df["label"].unique()))

label
0    886
1    823
2    834
Name: count, dtype: int64

Unique labels:
[np.int64(0), np.int64(1), np.int64(2)]


In [24]:
RESULTS_PATH = os.path.join(
    PROJECT_PATH,
    "results",
    "efficientnet_b3_mc_dropout_seed42_10passes.csv"
)

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

mc_results.to_csv(RESULTS_PATH, index=False)

print("Saved successfully!")
print(RESULTS_PATH)

Saved successfully!
/content/drive/MyDrive/MRI_Generalization/results/efficientnet_b3_mc_dropout_seed42_10passes.csv
